# Portfolio Replication — Full Comparison
Each section calls a single function from `utils/`. All outputs (figures, pickles) are saved automatically.

**Run order:** Data Loader → Linear Models → Kalman → NN → Evaluate

In [1]:
import sys, pickle, logging
from pathlib import Path

# ── path so imports resolve from the project root ────────────────────────
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import setup_logging
setup_logging(logging.INFO)

PICKLE_DIR = ROOT / 'data' / 'picklefiles'


## 1 · Data Loader
Loads prices, computes weekly returns, constructs the **monster index** (50% HFRX, 25% MSCI World, 25% Global Agg Bond), and saves 5 diagnostic plots.

In [2]:
from utils.data_loader import run_data_loader

data = run_data_loader(
    filepath=ROOT / 'data' / 'Dataset3_PortfolioReplicaStrategy.xlsx'
)


18:34:06 | INFO     | utils.data_loader         — ============================================================
18:34:06 | INFO     | utils.data_loader         — DATA LOADER — START
18:34:06 | INFO     | utils.data_loader         — ============================================================
18:34:06 | INFO     | utils.data_loader         — Reading Excel file: C:\Users\giuli\Repositories\fintech-group-work\BusinessCase3\data\Dataset3_PortfolioReplicaStrategy.xlsx (sheet=0)
18:34:06 | INFO     | utils.data_loader         — Loaded 705 observations | 2007-10-23 → 2021-04-20
18:34:06 | INFO     | utils.data_loader         — Computing weekly returns from price levels
18:34:06 | INFO     | utils.data_loader         — Returns shape: (704, 15)
18:34:06 | INFO     | utils.data_loader         — Monster index composition: HFRXGL Index=50% | MXWO Index=25% | LEGATRUU Index=25%
18:34:06 | INFO     | utils.data_loader         — X shape: (704, 11) | y shape: (704,)
18:34:06 | INFO     | utils.data_loa

In [3]:
# Quick sanity check — shape and date range
print('Prices  :', data['prices'].shape,  '|', data['prices'].index[0].date(), '→', data['prices'].index[-1].date())
print('Returns :', data['returns'].shape)
print('X       :', data['X'].shape,  '(futures features)')
print('y       :', data['y'].shape,  '(monster index)')
data['y'].describe().round(5)


Prices  : (705, 15) | 2007-10-23 → 2021-04-20
Returns : (704, 15)
X       : (704, 11) (futures features)
y       : (704,) (monster index)


count    704.00000
mean       0.00049
std        0.00878
min       -0.06694
25%       -0.00361
50%        0.00143
75%        0.00552
max        0.03377
Name: MonsterIndex, dtype: float64

*Plots saved to `outputs/01_price_series.png` … `05_return_stats.png`.*

## 2 · Linear Models (OLS / Ridge / LASSO / Elastic Net)
> **TODO** — `utils/run_linear_models.py` not yet created.

Once available, this cell will call:
```python
from utils.run_linear_models import run_linear_models
linear_results = run_linear_models(data['X'], data['y'])
```

In [ ]:
# Load pre-computed linear model results if the pickle exists
linear_pkl = PICKLE_DIR / 'linear_results.pkl'
if linear_pkl.exists():
    with open(linear_pkl, 'rb') as fh:
        linear_data = pickle.load(fh)
    linear_results = linear_data['best_results']   # dict: model_name → result_dict
    print('Loaded linear model results:', list(linear_results.keys()))
else:
    linear_results = {}
    print('No linear model pickle found — skipping.')


## 3 · Kalman Filter
> **TODO** — `utils/run_kalman.py` not yet created.

Once available:
```python
from utils.run_kalman import run_kalman
kalman_results = run_kalman(data['X'], data['y'])
```

In [ ]:
kalman_pkl = PICKLE_DIR / 'kalman_results.pkl'
if kalman_pkl.exists():
    with open(kalman_pkl, 'rb') as fh:
        kalman_data = pickle.load(fh)
    kalman_result = kalman_data['best_result']
    print('Loaded Kalman results')
else:
    kalman_result = None
    print('No Kalman pickle found — skipping.')


## 4 · Neural Network
Trains MLP and LSTM weight-generator networks across multiple hyperparameter configurations. Evaluated on the held-out test set (last 25 % of data).

**Loss**: MSE(replica, target) + λ·L1(weights)  
**Post-processing**: VaR scaling to respect UCITS 1M VaR(99%) ≤ 20%

In [4]:
from utils.run_nn import run_nn, DEFAULT_CONFIGS

nn_output = run_nn(
    X=data['X'],
    y=data['y'],
    configs=DEFAULT_CONFIGS,   # or pass a custom list
    train_frac=0.60,
    val_frac=0.15,
    max_var_threshold=0.20,
)


18:34:16 | INFO     | utils.run_nn              — ============================================================
18:34:16 | INFO     | utils.run_nn              — NN EXPERIMENT — START
18:34:16 | INFO     | utils.run_nn              — ============================================================
18:34:16 | INFO     | utils.run_nn              — Data split: train=422 | val=106 | test=176 (total=704)
18:34:16 | INFO     | utils.run_nn              — --------------------------------------------------
18:34:16 | INFO     | utils.run_nn              — Config 1/4: MLP_w26_h64x32_l10.0
18:34:16 | INFO     | utils.run_nn              — Device: cpu
18:34:18 | INFO     | utils.run_nn              — Epoch    1/300 | train=0.000194 | val=0.000046 | patience 0/30
18:34:18 | INFO     | utils.run_nn              — Epoch   25/300 | train=0.000011 | val=0.000009 | patience 1/30
18:34:19 | INFO     | utils.run_nn              — Epoch   50/300 | train=0.000009 | val=0.000009 | patience 22/30
18:34:19 | INFO

In [5]:
# Per-config metrics summary
nn_output['metrics_df'][[
    'tracking_error', 'information_ratio',
    'correlation', 'sharpe_replica', 'max_dd_replica'
]].round(4)


,tracking_error,information_ratio,correlation,sharpe_replica,max_dd_replica
model,,,,,
MLP_w26_h64x32_l10.0,0.0370,-0.3553,0.8382,0.8240,0.0941
MLP_w52_h64x32_l10.001,0.0507,-0.6023,0.7064,0.8307,0.0554
MLP_w26_h128x64x32_l10.001,0.0490,-0.6008,0.7085,0.7354,0.0627
LSTM_w52_h64_l10.0,0.0354,-0.2780,0.8521,0.8449,0.0981


*Training curves, weight plots, and VaR charts saved to `outputs/nn_cfg*.png`.*

## 5 · Full Comparison
Aggregates all available model results into a single evaluation run. Generates 6 comparison plots and a metrics heatmap.

In [ ]:
from utils.evaluation import run_evaluation

# Build the combined results dict — add models as they become available
all_results = {}

# Linear models
all_results.update(linear_results)

# Kalman
if kalman_result is not None:
    all_results['Kalman'] = kalman_result

# Neural network (best config only — add all if you want)
all_results['NN_best'] = nn_output['best_result']

print('Models being compared:', list(all_results.keys()))


Models being compared: ['NN_best']


In [8]:
metrics = run_evaluation(all_results, save_prefix='final')
metrics[[
    'tracking_error', 'information_ratio',
    'correlation', 'sharpe_replica', 'max_dd_replica'
]].round(4)


18:35:37 | INFO     | utils.evaluation          — ============================================================
18:35:37 | INFO     | utils.evaluation          — EVALUATION — START  (1 models)
18:35:37 | INFO     | utils.evaluation          — ============================================================
18:35:37 | INFO     | utils.evaluation          — 
         tracking_error  information_ratio  correlation  sharpe_replica
model                                                                  
NN_best          0.0354             -0.278       0.8521          0.8449
18:35:38 | INFO     | utils.evaluation          — Figure saved → outputs\final_01_cumulative_returns.png
18:35:38 | INFO     | utils.evaluation          — Figure saved → outputs\final_02_tracking_metrics.png
18:35:39 | INFO     | utils.evaluation          — Figure saved → outputs\final_03_drawdowns.png
18:35:39 | INFO     | utils.evaluation          — Figure saved → outputs\final_04_scatter_returns.png
18:35:39 | INFO     | ut

c:\Users\giuli\Repositories\fintech-group-work\BusinessCase3\.venv\Lib\site-packages\seaborn\matrix.py:202: RuntimeWarning: All-NaN slice encountered
  vmin = np.nanmin(calc_data)
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase3\.venv\Lib\site-packages\seaborn\matrix.py:207: RuntimeWarning: All-NaN slice encountered
  vmax = np.nanmax(calc_data)


18:35:39 | INFO     | utils.evaluation          — Figure saved → outputs\final_06_metrics_heatmap.png
18:35:39 | INFO     | utils.evaluation          — Pickle saved → data\picklefiles\evaluation.pkl
18:35:39 | INFO     | utils.evaluation          — EVALUATION — DONE
18:35:39 | INFO     | utils.evaluation          — ============================================================


,tracking_error,information_ratio,correlation,sharpe_replica,max_dd_replica
model,,,,,
NN_best,0.0354,-0.278,0.8521,0.8449,0.0981


*Comparison plots saved to `outputs/final_01_*.png` … `final_06_*.png`.*

## 6 · Interpretation

Fill this section with your own analysis after running the full pipeline.

Suggested structure:
- Which model achieves the lowest **Tracking Error**?
- Which has the highest **Information Ratio**?
- How does **gross exposure** compare across models (leverage)?
- How does the NN perform vs Kalman — does extra complexity pay off?
- Where does each model fail (high drawdown periods, regime changes)?
